In [34]:
#imports
import sys
import numpy as np
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
%matplotlib inline

In [35]:
#do what we did above
all_df = ["comments1.csv","comments2.csv", "comments3.csv", "comments4.csv", "comments5.csv"]
dfs = []  # list to hold each dataframe
for fname in all_df:
    df = pd.read_csv("dataset/" + fname)
    
    trend = df[["textOriginal", "likeCount", "publishedAt"]]
    trend = trend.dropna(subset=["textOriginal"])
    trend['publishedAt'] = pd.to_datetime(trend['publishedAt'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
    first_date = trend["publishedAt"].min()
    trend["days_since_first"] = (trend["publishedAt"] - first_date).dt.days.astype(float)

    filtered_likes_df = trend[trend["likeCount"] > 0]
    average_like_count = filtered_likes_df["likeCount"].mean()
    successful_comments = trend[trend["likeCount"] >= average_like_count]
    dfs.append(successful_comments)

#vertically concat
total_df = pd.concat(dfs, ignore_index=True)

def log1p_mapper(like : float) -> float:
    values = np.log1p(like)
    return values

total_df["likeCountLn"] = total_df['likeCount'].apply(log1p_mapper)



In [36]:
sia = SentimentIntensityAnalyzer()
def extract_enhanced_features(text):
    """Extract comprehensive features for comment quality assessment"""
    if pd.isna(text) or not isinstance(text, str):
        return {
            'comment_length': 0,
            'word_count': 0,
            'has_emoji': 0,
            'has_mention': 0,
            'has_hashtag': 0,
            'has_url': 0,
            'exclamation_count': 0,
            'question_count': 0,
            'caps_ratio': 0,
            'sentiment_compound': 0,
            'sentiment_positive': 0,
            'sentiment_negative': 0,
            'sentiment_neutral': 0
        }
    
    # Basic text features
    comment_length = len(text)
    word_count = len(text.split())
    
    # Pattern detection
    has_mention = 1 if '@' in text else 0
    has_hashtag = 1 if '#' in text else 0
    has_url = 1 if bool(re.search(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', text)) else 0
    
    # Punctuation analysis
    exclamation_count = text.count('!')
    question_count = text.count('?')
    
    # Caps analysis
    if len(text) > 0:
        caps_ratio = sum(1 for c in text if c.isupper()) / len(text)
    else:
        caps_ratio = 0
    
    # Sentiment analysis
    sentiment_scores = sia.polarity_scores(text)
    
    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'has_mention': has_mention,
        'has_hashtag': has_hashtag,
        'has_url': has_url,
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'caps_ratio': caps_ratio,
        'sentiment_compound': sentiment_scores['compound'],
        'sentiment_positive': sentiment_scores['pos'],
        'sentiment_negative': sentiment_scores['neg'],
        'sentiment_neutral': sentiment_scores['neu']
    }

In [37]:
print("Extracting enhanced features...")
feature_list = []
for text in total_df['textOriginal']:
    features = extract_enhanced_features(text)
    feature_list.append(features)

# Add features to dataframe
feature_df = pd.DataFrame(feature_list)
for col in feature_df.columns:
    total_df[col] = feature_df[col]

Extracting enhanced features...


In [38]:
def calculate_quality_score(row):
    max_likes = total_df['likeCount'].max()
    engagement_score = min(row['likeCount'] / (max_likes * 0.1), 1.0)  # Cap at reasonable level

    sentiment_score = max(0, row['sentiment_compound'] + 1) / 2  # Convert from [-1,1] to [0,1]
    
    # Length quality (moderate length is better)
    if row['word_count'] < 3:
        length_score = 0.3  # Too short
    elif row['word_count'] > 50:
        length_score = 0.7  # Too long
    else:
        length_score = 1.0  # Good length
    
    # Interaction quality (comments with mentions/hashtags show engagement)
    interaction_score = 0.5  # Base score
    if row['has_mention']:
        interaction_score += 0.2
    if row['has_hashtag']:
        interaction_score += 0.2
    if row['has_url']:
        interaction_score -= 0.1  # URLs might be promotional
    interaction_score = min(interaction_score, 1.0)  # Cap at 1.0
    
    # Expressiveness (appropriate use of punctuation shows engagement)
    if 1 <= row['exclamation_count'] <= 3:
        express_score = 1.0
    elif row['question_count'] >= 1:
        express_score = 0.8  # Questions show engagement
    elif row['exclamation_count'] > 3:
        express_score = 0.5  # Too many exclamations might be spam
    else:
        express_score = 0.6  # Neutral
    
    # Caps penalty (excessive caps indicate shouting/spam)
    caps_penalty = 1.0
    if row['caps_ratio'] > 0.5:  # More than 50% caps
        caps_penalty = 0.3
    elif row['caps_ratio'] > 0.3:  # More than 30% caps
        caps_penalty = 0.7
    
    # Weighted combination
    quality_score = (
        engagement_score * 0.35 +      
        sentiment_score * 0.25 +
        length_score * 0.15 +
        interaction_score * 0.15 +
        express_score * 0.1
    ) * caps_penalty  # Apply caps penalty
    
    return round(quality_score, 3)

#add a quality score for every comment through calculations
total_df['quality_score'] = total_df.apply(calculate_quality_score, axis=1)

In [39]:
def assign_quality_tier(score):
    if score >= 0.7:
        return 'High'
    elif score >= 0.4:
        return 'Medium'
    else:
        return 'Low'

total_df['quality_tier'] = total_df['quality_score'].apply(assign_quality_tier)

In [40]:
#machine learning imports
from lightgbm import LGBMRegressor   
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

In [41]:
# features and label

enhanced_features = [
    'days_since_first', 'comment_length', 'word_count', 'has_mention', 
    'has_hashtag', 'exclamation_count', 'question_count', 'caps_ratio',
    'sentiment_compound', 'sentiment_positive', 'sentiment_negative', 'likeCountLn'
]

X = total_df[["textOriginal"] + enhanced_features] #feature
y = total_df["quality_score"]    #label

# text + numeric preprocessing
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1,2), min_df=5) #https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
#only transform the features for training, they become a matrix
pre = ColumnTransformer([
    ("txt", tfidf, "textOriginal"),
    ("num", "passthrough", enhanced_features),
])

#model receives feature matrix and y_train to create model
model = LGBMRegressor(
    n_estimators=400, #num of trees
    learning_rate=0.05, #slower learning rate per tree but more stable
    subsample=0.85, #each tree takes 85 percent of the training dataset to avoid overfitting
    colsample_bytree=0.85,
    reg_lambda=0.5,
    n_jobs=-1
)

pipe = Pipeline([
    ("pre", pre),
    ("model", model),
])

# train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)

# evaluate
y_pred = pipe.predict(X_val)
print("RMSE:", root_mean_squared_error(y_val, y_pred))
print("R^2:", r2_score(y_val, y_pred))

# add predictions back
total_df["pred"] = pipe.predict(X)
total_df["pred_quality_score"] = pipe.predict(X)
total_df["pred_quality_tier"] = total_df["pred_quality_score"].apply(assign_quality_tier)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.166105 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 149296
[LightGBM] [Info] Number of data points in the train set: 34366, number of used features: 3979
[LightGBM] [Info] Start training from score 0.448639
RMSE: 0.004636712952120963
R^2: 0.9971447817096611


In [42]:
total_df.head()

,textOriginal,likeCount,publishedAt,days_since_first,likeCountLn,comment_length,word_count,has_mention,has_hashtag,has_url,...,caps_ratio,sentiment_compound,sentiment_positive,sentiment_negative,sentiment_neutral,quality_score,quality_tier,pred,pred_quality_score,pred_quality_tier
0,How do you achieve that slick back? 🧐,684,2025-01-28 09:03:05+00:00,1850.0,6.529419,37,8,0,0,0,...,0.027027,0.0000,0.000,0.000,1.000,0.435,Medium,0.436343,0.436343,Medium
1,"“If it rains, I’m ruined” was so REALL😭😮😢",88,2024-07-13 05:19:05+00:00,1651.0,4.488636,41,8,0,0,0,...,0.170732,0.0000,0.000,0.000,1.000,0.411,Medium,0.410452,0.410452,Medium
2,Can we please go back to Classic beauty?,75,2024-06-11 04:17:52+00:00,1619.0,4.330733,40,8,0,0,0,...,0.050000,0.7269,0.504,0.000,0.496,0.521,Medium,0.521315,0.521315,Medium
3,POPULAR..YOURE GONNA BE POPULARR!,701,2024-11-25 15:03:24+00:00,1787.0,6.553933,33,4,0,0,0,...,0.818182,0.0000,0.000,0.000,1.000,0.137,Low,0.136981,0.136981,Low
4,"Bro my skincare routine is soap, water and a t...",808,2023-04-29 16:40:35+00:00,1211.0,6.695799,120,24,0,0,0,...,0.016667,0.7783,0.311,0.081,0.608,0.513,Medium,0.514796,0.514796,Medium


In [43]:
feature_names = pipe.named_steps['pre'].get_feature_names_out()
importances = pipe.named_steps['model'] .feature_importances_

feature_importance = (pd.DataFrame({'feature': feature_names, 'importance': importances})
        .sort_values('importance', ascending=False))

print("\nTop 20 Most Important Features for Quality Prediction:")
print(feature_importance.head(20))


Top 20 Most Important Features for Quality Prediction:
                      feature  importance
4008  num__sentiment_compound        1857
4011         num__likeCountLn        1514
4002          num__word_count         936
4005   num__exclamation_count         921
4007          num__caps_ratio         797
4009  num__sentiment_positive         495
4001      num__comment_length         489
4010  num__sentiment_negative         452
4006      num__question_count         331
4003         num__has_mention         307
3246            txt__the make         163
1544                  txt__im         160
4000    num__days_since_first         139
4004         num__has_hashtag         111
2420             txt__omg she         108
1506               txt__https         101
86              txt__adorable          92
3205                 txt__the          92
3854            txt__would be          74
3519               txt__trust          72


In [44]:
print(type(y_train))
print(y_train[:10])

<class 'pandas.core.series.Series'>
28675    0.468
35360    0.410
8920     0.451
3100     0.530
11247    0.416
5182     0.457
1959     0.526
27290    0.568
33533    0.411
10504    0.411
Name: quality_score, dtype: float64
